# Neural Network Training

## Imports

In [ ]:
from pathlib import Path
import ctypes
import os
import site
from datetime import datetime

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Preload CUDA shared libraries from pip-installed NVIDIA wheels before TensorFlow import.
site_packages_dir = Path(site.getsitepackages()[0])
nvidia_root = site_packages_dir / "nvidia"
cuda_lib_dirs = [p for p in nvidia_root.rglob("*") if p.is_dir() and p.name in {"lib", "lib64"}]
for lib_dir in cuda_lib_dirs:
    for lib_file in sorted(lib_dir.glob("*.so*")):
        try:
            ctypes.CDLL(str(lib_file), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

import tensorflow as tf
from tensorflow import keras
from keras import layers, regularizers
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from enum import Enum
from dataclasses import dataclass

## Helper Enums and Data Classes

In [ ]:
class DIMENSION(Enum):
    K_9 = 9
    K_12 = 12
    K_21 = 21

class Classes(Enum):
    NO_DIABETES = 0
    PREDIABETES = 1
    DIABETES = 2

@dataclass
class DatasetSplit:
    X_train: pd.DataFrame
    y_train: pd.Series
    X_val: pd.DataFrame
    y_val: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series

@dataclass
class HyperParamVersion:
    learning_rate: float
    hidden_layers: int
    start_neurons: int
    dropout_rate: float
    l2_lambda: float
    epochs: int
    batch_size: int

class ValMacroF1Callback(keras.callbacks.Callback):
    def __init__(self, x_val, y_val_labels):
        super().__init__()
        self.x_val = x_val
        self.y_val_labels = np.asarray(y_val_labels)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_val_pred = np.argmax(self.model.predict(self.x_val, verbose=0), axis=-1)
        logs["val_f1_macro"] = f1_score(self.y_val_labels, y_val_pred, average="macro", zero_division=0)

## Load Datasets

In [ ]:
# Helper functions to load datasets
UNDERSAMPLING_DATASET = lambda k: Path(f"datasets/undersampling/undersampling_k{str(k)}.csv")
SMOTE_DATASET = lambda k: Path(f"datasets/oversampling/Smote_k{str(k)}.csv")

base_dataset = pd.read_csv("datasets/BaseDataset.csv")
undersampling_datasets = {k: pd.read_csv(UNDERSAMPLING_DATASET(k.value)) for k in DIMENSION}
smote_datasets = {k: pd.read_csv(SMOTE_DATASET(k.value)) for k in DIMENSION}

print("Base Dataset Shape:", base_dataset.shape)
for k in DIMENSION:
    print(f"Undersampling Dataset (k={k.value}) Shape:", undersampling_datasets[k].shape, end="\t | \t")
    print(f"Smote Dataset (k={k.value}) Shape:", smote_datasets[k].shape)

## Split datasets

In [ ]:
"""Splits them to to train, validation, and test sets"""

# Configurations
RANDOM_STATE = 42
TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.1
TEST_RATIO = 0.2
TARGET_COLUMN = "Diabetes_012"

def split_dataset(dataset):
    X = dataset.drop(columns=[TARGET_COLUMN])
    y = dataset[TARGET_COLUMN]

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1-TRAIN_RATIO, random_state=RANDOM_STATE, stratify=y)
    validation_test_ratio = VALIDATION_RATIO / (VALIDATION_RATIO + TEST_RATIO)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=validation_test_ratio, random_state=RANDOM_STATE, stratify=y_temp)

    return DatasetSplit(X_train, y_train, X_val, y_val, X_test, y_test)

base_dataset = split_dataset(base_dataset)
undersampling_datasets = {k: split_dataset(undersampling_datasets[k]) for k in DIMENSION}
smote_datasets = {k: split_dataset(smote_datasets[k]) for k in DIMENSION}

print("Base Dataset Split Shapes:")
print("X_train:", base_dataset.X_train.shape, "y_train:", base_dataset.y_train.shape)
print("X_val:", base_dataset.X_val.shape, "y_val:", base_dataset.y_val.shape)
print("X_test:", base_dataset.X_test.shape, "y_test:", base_dataset.y_test.shape)
print()
print("\t\t\tUndersampling", end="\t\t\t  \t\t\t")
print("Smote")
for k in DIMENSION:
    print(f"\t\t\t\t\t Dataset (k={k.value}) Split Shapes")
    print("X_train:", undersampling_datasets[k].X_train.shape, "y_train:", undersampling_datasets[k].y_train.shape, end="\t\t\t | \t\t\t")
    print("X_train:", smote_datasets[k].X_train.shape, "y_train:", smote_datasets[k].y_train.shape)
    print("X_val:", undersampling_datasets[k].X_val.shape, "y_val:", undersampling_datasets[k].y_val.shape, end="\t\t\t | \t\t\t")
    print("X_val:", smote_datasets[k].X_val.shape, "y_val:", smote_datasets[k].y_val.shape)
    print("X_test:", undersampling_datasets[k].X_test.shape, "y_test:", undersampling_datasets[k].y_test.shape, end="\t\t\t | \t\t\t")
    print("X_test:", smote_datasets[k].X_test.shape, "y_test:", smote_datasets[k].y_test.shape)

    print("-----------------------------------")


## Scaling

In [ ]:
scaler = StandardScaler()

base_dataset.X_train = scaler.fit_transform(base_dataset.X_train)
base_dataset.X_val = scaler.transform(base_dataset.X_val)
base_dataset.X_test = scaler.transform(base_dataset.X_test)

for k in DIMENSION:
    undersampling_datasets[k].X_train = scaler.fit_transform(undersampling_datasets[k].X_train)
    undersampling_datasets[k].X_val = scaler.transform(undersampling_datasets[k].X_val)
    undersampling_datasets[k].X_test = scaler.transform(undersampling_datasets[k].X_test)

    smote_datasets[k].X_train = scaler.fit_transform(smote_datasets[k].X_train)
    smote_datasets[k].X_val = scaler.transform(smote_datasets[k].X_val)
    smote_datasets[k].X_test = scaler.transform(smote_datasets[k].X_test)

## Model Setup

In [ ]:
HIDDEN_LAYERS_RANGE = (2, 6)
START_NEURONS_OPTIONS = [32, 64, 96, 128, 192, 256, 512]
DROPOUT_RATE_RANGE = (0.05, 0.4)
L2_LAMBDA_OPTIONS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
LEARNING_RATE_OPTIONS = [1e-4, 3e-4, 5e-4, 8e-4, 1e-3, 3e-3, 5e-3, 1e-2, 5e-2, 1e-1]

def build_tuner_model(hp, input_shape):
    hidden_layers = hp.Int("hidden_layers", min_value=HIDDEN_LAYERS_RANGE[0], max_value=HIDDEN_LAYERS_RANGE[1], step=1)
    start_neurons = hp.Choice("start_neurons", values=START_NEURONS_OPTIONS)
    dropout_rate = hp.Float("dropout_rate", min_value=DROPOUT_RATE_RANGE[0], max_value=DROPOUT_RATE_RANGE[1], step=0.05)
    l2_lambda = hp.Choice("l2_lambda", values=L2_LAMBDA_OPTIONS)
    learning_rate = hp.Choice("learning_rate", values=LEARNING_RATE_OPTIONS)

    model = keras.Sequential()
    model.add(layers.Input(shape=input_shape))

    # Keep the first hidden layer width based on the sampled start_neurons option.
    current_neurons = start_neurons
    current_dropout = min(max(DROPOUT_RATE_RANGE[0], dropout_rate), DROPOUT_RATE_RANGE[1])

    for _ in range(hidden_layers):
        model.add(
            layers.Dense(
                current_neurons,
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_lambda)
            )
        )
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(current_dropout))
        current_neurons = max(16, current_neurons // 2)
        current_dropout = min(DROPOUT_RATE_RANGE[1], current_dropout + 0.05)

    model.add(layers.Dense(3, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## KerasTuner Search Space

In [ ]:
# KerasTuner / training configuration
MAX_TRIALS = 10
SEARCH_EPOCHS = 20
FINAL_TRAIN_EPOCHS = 50
EXECUTIONS_PER_TRIAL = 1
BATCH_SIZE = 16

## Training 

In [ ]:
def model_builder(hp):
        return build_tuner_model(hp, input_shape=X_train.shape[1:])


TENSORBOARD_LOG_DIR = Path("tensorboard_logs")

def train_model(data_set : DatasetSplit, model_name : str):
    print(f"Training model: {model_name}")

    X_train, y_train = data_set.X_train, data_set.y_train
    X_val, y_val = data_set.X_val, data_set.y_val
    X_test, y_test = data_set.X_test, data_set.y_test
    y_train_cat = keras.utils.to_categorical(y_train, num_classes=3)
    y_val_cat = keras.utils.to_categorical(y_val, num_classes=3)

    run_name = datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_log_dir = TENSORBOARD_LOG_DIR / f"{model_name}_{run_name}"
    tuner_dir = Path("tuner_runs") / f"{model_name}_{run_name}"

    tuner = kt.RandomSearch(
        hypermodel=model_builder,
        objective=kt.Objective("val_f1_macro", direction="max"),
        max_trials=MAX_TRIALS,
        executions_per_trial=EXECUTIONS_PER_TRIAL,
        directory=str(tuner_dir),
        project_name="base_dataset_tuning",
        overwrite=True
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    tensorboard_callback = keras.callbacks.TensorBoard(
        log_dir=str(tensorboard_log_dir),
        histogram_freq=1,
        write_graph=True
    )
    val_macro_f1_callback = ValMacroF1Callback(X_val, y_val)

    tuner.search(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=SEARCH_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping, val_macro_f1_callback],
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    print("Best Hyperparameters:")
    for name in ["hidden_layers", "start_neurons", "dropout_rate", "l2_lambda", "learning_rate"]:
        print(f"  {name}: {best_hp.get(name)}")

    best_model = tuner.hypermodel.build(best_hp)
    history = best_model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=FINAL_TRAIN_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping, tensorboard_callback, val_macro_f1_callback],
        verbose=1
    )

    y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=-1)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision (weighted): {precision:.4f}")
    print(f"Recall (weighted): {recall:.4f}")
    print(f"F1 Micro: {f1_micro:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Diabetes", "Prediabetes", "Diabetes"]
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap="Blues", ax=ax, colorbar=False)
    plt.title("Confusion Matrix - Base Dataset")
    plt.tight_layout()
    plt.show()

    print(f"TensorBoard logs saved to: {tensorboard_log_dir}")
    print("Run `%load_ext tensorboard` then `%tensorboard --logdir tensorboard_logs` in a new cell.")
    return

## SMOTE training

## TensorBoard
Run the next cell to open TensorBoard and track training/validation loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard_logs